In [1]:
import argparse

import torch
import numpy as np

from geotransformer.utils.data import registration_collate_fn_stack_mode
from geotransformer.utils.torch import to_cuda, release_cuda
from geotransformer.utils.open3d import make_open3d_point_cloud, get_color, draw_geometries
from geotransformer.utils.registration import compute_registration_error

from config import make_cfg
from model import create_model

import open3d as o3d




Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
# Specify the desired GPU index (e.g., system GPU 1)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_device(device)

In [3]:
WEIGHTS = "../../output/with_aug/snapshots/epoch-40.pth.tar"


In [4]:
cfg = make_cfg()
model = create_model(cfg).cuda()
state_dict = torch.load(WEIGHTS)
model.load_state_dict(state_dict["model"])

<All keys matched successfully>

In [5]:
REF_NUM = 25

In [6]:
SRC_FILE = f"../../data/faces/demo/src_{REF_NUM}.npy"
REF_FILE = f"../../data/faces/demo/ref_{REF_NUM}.npy"
GT_FILE = f"../../data/faces/demo/gt_{REF_NUM}.npy"
MORPHED_FULL_FILE = f"../../data/faces/demo/morphed_full_{REF_NUM}.npy"

In [ ]:
def load_data():
    src_points = np.load(SRC_FILE)
    ref_points = np.load(REF_FILE)
    morphed_full_points = np.load(MORPHED_FULL_FILE)
    src_feats = np.ones_like(src_points[:, :1])
    ref_feats = np.ones_like(ref_points[:, :1])

    data_dict = {
        "ref_points": ref_points.astype(np.float32),
        "src_points": src_points.astype(np.float32),
        "ref_feats": ref_feats.astype(np.float32),
        "src_feats": src_feats.astype(np.float32),
        "morphed_full": morphed_full_points.astype(np.float32),
        "gt_z": np.zeros((32, 100), dtype=np.float32) 
    }

    if GT_FILE is not None:
        transform = np.load(GT_FILE)
        data_dict["transform"] = transform.astype(np.float32)

    return data_dict

def open3d_webrtc_draw(geometries):
    o3d.visualization.draw(geometries)
   

In [8]:
data_dict = load_data()

In [9]:
data_dict.keys()

dict_keys(['ref_points', 'src_points', 'ref_feats', 'src_feats', 'morphed_full', 'gt_z', 'transform'])

In [10]:

# prepare data
neighbor_limits = [38, 36, 36, 38]  # default setting in 3DMatch
data_dict = registration_collate_fn_stack_mode(
    [data_dict], cfg.backbone.num_stages, cfg.backbone.init_voxel_size, cfg.backbone.init_radius, neighbor_limits
)


# prediction
data_dict = to_cuda(data_dict)
output_dict = model(data_dict)
data_dict = release_cuda(data_dict)
output_dict = release_cuda(output_dict)

# get results
ref_points = output_dict["ref_points"]
src_points = output_dict["src_points"]
estimated_transform = output_dict["estimated_transform"]
transform = data_dict["transform"]

# visualization
ref_pcd = make_open3d_point_cloud(ref_points)
ref_pcd.estimate_normals()
ref_pcd.paint_uniform_color(get_color("custom_blue"))
src_pcd = make_open3d_point_cloud(src_points)
src_pcd.estimate_normals()
src_pcd.paint_uniform_color(get_color("custom_yellow"))
o3d.visualization.draw_plotly([ref_pcd, src_pcd])


In [11]:
estimated_t_src_pcd = src_pcd.transform(estimated_transform)
o3d.visualization.draw_plotly([ref_pcd, estimated_t_src_pcd])

# compute error
rre, rte = compute_registration_error(transform, estimated_transform)
print(f"RRE(deg): {rre:.3f}, RTE(m): {rte:.3f}")

RRE(deg): 4.730, RTE(m): 0.017


In [ ]:
new_src_pcd = make_open3d_point_cloud(src_points)
new_src_pcd.transform(transform)
o3d.visualization.draw_plotly([estimated_t_src_pcd, new_src_pcd])

In [13]:
o3d.visualization.draw_plotly([ref_pcd, new_src_pcd])

In [14]:
pred_morphed_data = output_dict["morphed_full"]

if torch.is_tensor(pred_morphed_data):
    pred_morphed_data = pred_morphed_data.detach().cpu().numpy()
if pred_morphed_data.ndim == 3:
    pred_morphed_data = pred_morphed_data.squeeze(0)

pred_morphed_pcd = o3d.geometry.PointCloud()
pred_morphed_pcd.points = o3d.utility.Vector3dVector(pred_morphed_data)
pred_morphed_pcd.estimate_normals()
pred_morphed_pcd.paint_uniform_color([0.0, 1.0, 0.0]) # Green = Predicted Model Output

recon_gt_data = output_dict["recon_gt_points"]
if torch.is_tensor(recon_gt_data):
    recon_gt_data = recon_gt_data.detach().cpu().numpy()
if recon_gt_data.ndim == 3:
    recon_gt_data = recon_gt_data.squeeze(0)

recon_gt_pcd = o3d.geometry.PointCloud()
recon_gt_pcd.points = o3d.utility.Vector3dVector(recon_gt_data)
recon_gt_pcd.estimate_normals()
recon_gt_pcd.paint_uniform_color([0.0, 0.0, 1.0]) # Blue = GT PCA Reconstruction

print("Visualizing: Predicted Morphed Shape (Green) vs Ground Truth PCA Reconstruction (Blue)")
o3d.visualization.draw_plotly([pred_morphed_pcd, recon_gt_pcd])


Visualizing: Predicted Morphed Shape (Green) vs Ground Truth PCA Reconstruction (Blue)


In [15]:
estimated_t_src_pcd.paint_uniform_color([0, 0, 0.5])
o3d.visualization.draw_plotly([pred_morphed_pcd, estimated_t_src_pcd])

In [16]:
print("Available keys in output_dict:", output_dict.keys())
if 'matching_scores' in output_dict:
    scores = output_dict['matching_scores']
    print(f"Max matching score: {scores.max().item():.4f}")
    print(f"Mean matching score: {scores.mean().item():.4f}")
    if scores.max() < 0.1:
        print("Warning: Model has very low confidence in these matches!")

Available keys in output_dict: dict_keys(['z_coefficients', 'morphed_full', 'recon_gt_points', 'ref_points_c', 'src_points_c', 'ref_points_f', 'src_points_f', 'ref_points', 'src_points', 'gt_node_corr_indices', 'gt_node_corr_overlaps', 'ref_feats_c', 'src_feats_c', 'ref_feats_f', 'src_feats_f', 'ref_node_corr_indices', 'src_node_corr_indices', 'ref_node_corr_knn_points', 'src_node_corr_knn_points', 'ref_node_corr_knn_masks', 'src_node_corr_knn_masks', 'matching_scores', 'ref_corr_points', 'src_corr_points', 'corr_scores', 'estimated_transform'])
Max matching score: 2.6993
Mean matching score: -926068768768.0000


In [17]:
ply_path = "3099.ply"  # Nicolas sample

ply_pcd = o3d.io.read_point_cloud(ply_path)
ply_pcd.estimate_normals()
ply_pcd.paint_uniform_color([0.8, 0.2, 0.2]) 

print(f"Visualizing {ply_path}...")

o3d.visualization.draw_plotly([ply_pcd])


Visualizing 3099.ply...


In [20]:
import torch
import numpy as np
import open3d as o3d
import copy
from geotransformer.utils.data import registration_collate_fn_stack_mode
from geotransformer.utils.torch import to_cuda, release_cuda
from geotransformer.utils.open3d import make_open3d_point_cloud
from config import make_cfg
from model import create_model

# --- CONFIG ---
WEIGHTS = "../../output/with_aug/snapshots/epoch-40.pth.tar"
SRC_FILE = "plank.npy" 
REF_FILE = "../../data/faces/demo/ref_3.npy"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = make_cfg()
model = create_model(cfg).cuda()
model.load_state_dict(torch.load(WEIGHTS)["model"])
model.eval()

def preprocess_global(src_points, ref_points):
    # 1. SHIFT TO ORIGIN (Rough)
    ref_points = ref_points - np.median(ref_points, axis=0)
    src_points = src_points - np.median(src_points, axis=0)
    
    # 2. SCALE MATCHING (Keep your existing semantic scale)
    ref_diag = np.linalg.norm(ref_points.max(axis=0) - ref_points.min(axis=0))
    src_diag = np.linalg.norm(src_points.max(axis=0) - src_points.min(axis=0))
    src_points = src_points * ((ref_diag * 0.55) / src_diag)

    # 3. SEMANTIC NOSE ALIGNMENT (The Secret Sauce)
    # On a standard face model, the nose is usually the point furthest forward 
    # (max or min on the Z-axis, depending on orientation). 
    # Let's find the "tip" and shift the origin there.
    ref_tip_idx = np.argmin(ref_points[:, 2]) # Assuming Z is forward
    src_tip_idx = np.argmin(src_points[:, 2])
    
    ref_points = ref_points - ref_points[ref_tip_idx]
    src_points = src_points - src_points[src_tip_idx]
    
    return src_points.astype(np.float32), ref_points.astype(np.float32)

def run_global_stage_2_alignment(morphed_pts, src_pts):
    """
    Global Alignment Strategy: Uses superpoint matches to find arbitrary poses.
    """
    data_dict = {
        "ref_points": morphed_pts.astype(np.float32),
        "src_points": src_pts.astype(np.float32),
        "ref_feats": np.ones((len(morphed_pts), 1), dtype=np.float32),
        "src_feats": np.ones((len(src_pts), 1), dtype=np.float32),
        "transform": np.eye(4, dtype=np.float32),
        "gt_z": np.zeros((32, 100), dtype=np.float32),
        "morphed_full": np.zeros_like(morphed_pts, dtype=np.float32)
    }
    
    neighbor_limits = [38, 36, 36, 38]
    collated = registration_collate_fn_stack_mode([data_dict], 4, 0.025, 0.05, neighbor_limits)
    
    with torch.no_grad():
        output = release_cuda(model(to_cuda(collated)))
    
    src_corr = output['src_corr_points']
    ref_corr = output['ref_corr_points']
    
    # GLOBAL RANSAC: No spatial mask here to allow for arbitrary rotations
    num_corrs = len(src_corr)
    indices = np.arange(num_corrs).reshape(-1, 1)
    corres = o3d.utility.Vector2iVector(np.hstack([indices, indices]))

    print(f"Executing Global Search on {num_corrs} correspondences...")
    
    # Increased max_correspondence_distance (10cm) for global pose recovery
    ransac = o3d.pipelines.registration.registration_ransac_based_on_correspondence(
        make_open3d_point_cloud(src_corr), 
        make_open3d_point_cloud(ref_corr), 
        corres,
        max_correspondence_distance=0.1, 
        estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
        ransac_n=4,
        criteria=o3d.pipelines.registration.RANSACConvergenceCriteria(4000000, 500)
    )
    return ransac.transformation, len(ransac.correspondence_set)

# --- MAIN EXECUTION ---
raw_src = np.load(SRC_FILE)
raw_ref = np.load(REF_FILE)

# Remove the 0/180 loop; let the network and global RANSAC find the pose
src_pts, ref_pts = preprocess_global(raw_src, raw_ref)

# Pass 1: Extract Morphed Reference (Standard)
data_dict = {
    "ref_points": ref_pts.astype(np.float32), 
    "src_points": src_pts.astype(np.float32), 
    "ref_feats": np.ones((len(ref_pts), 1), dtype=np.float32), 
    "src_feats": np.ones((len(src_pts), 1), dtype=np.float32), 
    "transform": np.eye(4, dtype=np.float32), 
    "gt_z": np.zeros((32, 100), dtype=np.float32), 
    "morphed_full": np.zeros_like(ref_pts, dtype=np.float32)
}

neighbor_limits = [38, 36, 36, 38]
collated = registration_collate_fn_stack_mode([data_dict], 4, 0.025, 0.05, neighbor_limits)

with torch.no_grad():
    output = release_cuda(model(to_cuda(collated)))

morphed_pts = output['morphed_full']

# Stage 2: Global Rigid Alignment
best_t, inliers = run_global_stage_2_alignment(morphed_pts, src_pts)

# --- FINAL VISUALIZATION ---
print(f"Global Alignment Complete! Inliers: {inliers}")
morphed_pcd = make_open3d_point_cloud(morphed_pts).paint_uniform_color([1, 0, 0])
src_pcd = make_open3d_point_cloud(src_pts).paint_uniform_color([1, 0.7, 0])
src_pcd.transform(best_t)

o3d.visualization.draw_plotly([morphed_pcd, src_pcd])

Executing Global Search on 2346 correspondences...
Global Alignment Complete! Inliers: 1590


In [21]:
# --- Extract the Morphed Output ---
morphed_points = output_dict["morphed_full"]  # <--- Removed .cpu().numpy()

# --- Plotly Visualization ---
import plotly.graph_objects as go

fig = go.Figure()

# 1. The Original Source Scan (Yellow)
fig.add_trace(go.Scatter3d(
    x=src_points[:, 0], y=src_points[:, 1], z=src_points[:, 2],
    mode='markers',
    marker=dict(size=2, color='orange'),
    name="Source Scan"
))

# 2. The Morphed Reference Face (Red)
fig.add_trace(go.Scatter3d(
    x=morphed_points[:, 0], y=morphed_points[:, 1], z=morphed_points[:, 2],
    mode='markers',
    marker=dict(size=2, color='red'),
    name="Network's Morphed Reference"
))

fig.update_layout(
    scene=dict(aspectmode='data'), 
    title="Debugging the PCA Morphing Pass"
)
fig.show()

In [22]:
# Get the morphed and src 
morphed_points = output_dict["morphed_full"] 
src_points = output_dict["src_points"]

morphed_pcd = make_open3d_point_cloud(morphed_points)
morphed_pcd.estimate_normals()
morphed_pcd.paint_uniform_color([1.0, 0.0, 0.0]) # Red

src_pcd = make_open3d_point_cloud(src_points)
src_pcd.estimate_normals()
src_pcd.paint_uniform_color([1.0, 0.706, 0.0]) # Yellow

# Bypass Pass 2 (alignment) and use Open3D ICP 
threshold = 0.05 # 5cm search radius
trans_init = np.eye(4) # Start from their current (pre-aligned) positions

reg_p2p = o3d.pipelines.registration.registration_icp(
    src_pcd, morphed_pcd, threshold, trans_init,
    o3d.pipelines.registration.TransformationEstimationPointToPoint(),
    o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=200)
)

print("ICP Transform Matrix:")
print(reg_p2p.transformation)

# Visualize Alignment
src_pcd.transform(reg_p2p.transformation)

print("Visualizing: Network's Morphed Face (Red) and ICP-Aligned Source (Yellow)")
o3d.visualization.draw_plotly([morphed_pcd, src_pcd])

ICP Transform Matrix:
[[ 0.9985199  -0.05280095 -0.01304068  0.01265865]
 [ 0.05262417  0.9985225  -0.01354709 -0.03212358]
 [ 0.01373671  0.01284078  0.99982319 -0.00416245]
 [ 0.          0.          0.          1.        ]]
Visualizing: Network's Morphed Face (Red) and ICP-Aligned Source (Yellow)
